<a href="https://colab.research.google.com/github/Dani0601/pm-turi2-prepocessing-danihidayat/blob/main/PM_P4_DaniHidayat_2488010016.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Dataset karyawan (sengaja mengandung masalah kualitas)
data = {
    'usia'      : [25, 32, np.nan, 45, 28, 51, 38, np.nan, 29, 41],
    'pendapatan': [4_000_000, 7_500_000, 5_200_000, 12_000_000, 4_800_000,
                   15_000_000, 8_100_000, 6_300_000, np.nan, 9_900_000],
    'pendidikan': ['SMA','S1','SMA','S2','SMA','S2','S1','S1','SMA','S2'],  # ordinal
    'kota'      : ['Bandung','Jakarta','Bandung','Surabaya','Jakarta',
                   'Surabaya','Jakarta','Bandung','Bandung','Jakarta'],    # nominal
    'membeli'   : ['Tidak','Ya','Tidak','Ya','Tidak','Ya','Ya','Tidak','Tidak','Ya']  # target
}

df = pd.DataFrame(data)
df['usia'] = df['usia'].fillna(df['usia'].mean())
df['pendapatan'] = df['pendapatan'].fillna(df['pendapatan'].median())
df['departemen'] = ['IT','HR','IT','HR','IT','HR','IT','HR','IT','HR']
print(df)

     usia  pendapatan pendidikan      kota membeli departemen
0  25.000   4000000.0        SMA   Bandung   Tidak         IT
1  32.000   7500000.0         S1   Jakarta      Ya         HR
2  36.125   5200000.0        SMA   Bandung   Tidak         IT
3  45.000  12000000.0         S2  Surabaya      Ya         HR
4  28.000   4800000.0        SMA   Jakarta   Tidak         IT
5  51.000  15000000.0         S2  Surabaya      Ya         HR
6  38.000   8100000.0         S1   Jakarta      Ya         IT
7  36.125   6300000.0         S1   Bandung   Tidak         HR
8  29.000   7500000.0        SMA   Bandung   Tidak         IT
9  41.000   9900000.0         S2   Jakarta      Ya         HR


In [2]:
X = df.drop(columns=['membeli'])
y = df['membeli']

In [3]:
X['pendidikan'] = X['pendidikan'].map({'SMA':0,'S1':1,'S2':2}) # ordinal
X = pd.get_dummies(X, columns=['departemen'], dtype=int) # nominal
y = y.map({'Tidak':0,'Ya':1})

In [4]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
X, y, test_size=0.3, random_state=42)

In [5]:
from sklearn.preprocessing import MinMaxScaler
num = ['usia','pendapatan','pendidikan']
scaler_minmax = MinMaxScaler()
X_train[num] = scaler_minmax.fit_transform(X_train[num])
X_test[num] = scaler_minmax.transform(X_test[num]) # cegah data leakage
print("Hasil describe() setelah menggunakan MinMaxScaler:")
print(X_train[num].describe())

Hasil describe() setelah menggunakan MinMaxScaler:
           usia  pendapatan  pendidikan
count  7.000000    7.000000    7.000000
mean   0.530357    0.398214    0.428571
std    0.349971    0.367808    0.449868
min    0.000000    0.000000    0.000000
25%    0.353125    0.125000    0.000000
50%    0.556250    0.287500    0.500000
75%    0.725000    0.625000    0.750000
max    1.000000    1.000000    1.000000


D. Latihan Mandiri
1. Median data Nan nya diisi menjadi nilai median: 35.0
   Mean data NaN nya diisi menjadi nilai mean: 36.125
2. MinMaxScaler (0–1) sebagai ganti StandardScaler; tampilkan describe().
3. Tambahkan kolom kategorikal baru dan terapkan one-hot encoding.
4.
- fit_transform digunakan pada data latih agar model scaler dapat mempelajari parameter statistik dari data tersebut (seperti nilai rata-rata dan simpangan baku pada StandardScaler, atau nilai minimum dan maksimum pada MinMaxScaler).
- Pada data uji, kita hanya menggunakan transform agar parameter statistik yang digunakan untuk mengubah data uji tetap berpatokan murni pada data latih.

- Jika kita melakukan fit_transform pada data uji, informasi dari data uji akan "bocor" ke dalam proses pelatihan (data leakage), yang berakibat pada penurunan performa generalisasi model saat dihadapkan pada data baru di dunia nyata.

E. REFLEKSI

1. Mengapa urutan 'split dulu, baru scaling' penting?

Urutan ini sangat penting untuk mencegah terjadinya kebocoran data (data leakage). Jika kita melakukan scaling (seperti mencari nilai rata-rata, standar deviasi, nilai minimum, atau maksimum) pada seluruh dataset sebelum dibagi, informasi statistik dari data uji (test set) akan ikut tercampur dan dipelajari oleh scaler. Akibatnya, data uji tidak lagi murni menjadi data baru yang belum pernah dilihat oleh sistem, yang dapat membuat evaluasi performa model menjadi terlalu optimis (overfitting) dan buruk saat diimplementasikan pada data dunia nyata. Oleh karena itu, data harus di-split terlebih dahulu, lalu proses fit scaler hanya dilakukan pada data latih (train), dan hasilnya digunakan untuk mentransformasi data latih serta data uji.

2. Kapan label encoding dan kapan one-hot encoding?

Label Encoding: Digunakan ketika variabel kategorikal bersifat ordinal (memiliki tingkatan atau urutan hierarki yang bermakna), contohnya tingkat pendidikan (SMA < S1 < S2) atau tingkat jabatan (Junior < Senior < Manajer). Hal ini karena label encoding mengubah kategori menjadi angka berurutan ($0, 1, 2, \dots$), sehingga hubungan urutannya tetap terjaga.

  One-Hot Encoding: Digunakan ketika variabel kategorikal bersifat nominal (tidak memiliki tingkatan atau hierarki sama sekali), contohnya nama kota (Jakarta, Bandung, Surabaya) atau departemen kerja (IT, HRD, Keuangan). Jika menggunakan label encoding pada data nominal, model machine learning bisa salah mengira bahwa angka yang lebih besar memiliki nilai/bobot yang lebih tinggi (misal: Surabaya $= 2$ dianggap lebih baik dari Jakarta $= 0$), sehingga one-hot encoding (yang memecahnya menjadi kolom biner terpisah) jauh lebih tepat agar tidak menimbulkan bias relasi angka.

3. Apa perbedaan normalisasi dan standardisasi?

Perbedaan utamanya terletak pada rumus matematika dan rentang nilai hasil transformasinya

Normalisasi (Contoh: Min-Max Scaling): Mengubah nilai fitur sedemikian rupa sehingga seluruh data berada dalam rentang skala tertentu, umumnya antara 0 hingga 1. Normalisasi sangat peka terhadap adanya nilai pencilan (outliers).   

Standardisasi (Contoh: StandardScaler): Mengubah data agar memiliki nilai rata-rata (mean) = 0 dan simpangan baku (standard deviation) = 1. Metode ini tidak membatasi data dalam rentang nilai minimum-maksimum tertentu, sehingga lebih tahan atau aman terhadap keberadaan outliers dibandingkan normalisasi.  